# Customer Churn Prediction Analysis

This notebook builds a complete customer churn prediction project in Jupyter. The goal is to predict whether a customer is likely to leave a company based on customer behavior, account information, and service usage.

The workflow includes:

1. Creating a realistic sample churn dataset
2. Inspecting and cleaning the data
3. Exploratory data analysis
4. Feature engineering
5. Preparing categorical and numerical variables
6. Training classification models
7. Evaluating model performance
8. Interpreting feature importance
9. Creating a simple churn-risk scoring table

This is designed as a portfolio-ready data science project. In a real business setting, the same structure could be used with actual customer data from a CRM, subscription platform, telecom company, bank, or SaaS product.

## 1. Import Libraries

We begin with the main Python libraries used for data analysis, visualization, preprocessing, modeling, and evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    RocCurveDisplay
)

np.random.seed(42)
pd.set_option("display.max_columns", 50)

## 2. Create a Sample Customer Churn Dataset

In a real project, this section would load a CSV or database table. For this portfolio example, we create a realistic synthetic dataset so the notebook can run without any external file.

The target variable is `churn`, where:

- `1` means the customer churned
- `0` means the customer stayed

In [ ]:
n = 2500

tenure_months = np.random.randint(1, 73, n)
monthly_charges = np.round(np.random.normal(75, 25, n), 2)
monthly_charges = np.clip(monthly_charges, 20, 150)
support_tickets = np.random.poisson(1.2, n)
late_payments = np.random.poisson(0.5, n)
contract_type = np.random.choice(["Month-to-month", "One year", "Two year"], size=n, p=[0.55, 0.25, 0.20])
internet_service = np.random.choice(["Fiber optic", "DSL", "None"], size=n, p=[0.52, 0.38, 0.10])
payment_method = np.random.choice(["Electronic check", "Credit card", "Bank transfer", "Mailed check"], size=n, p=[0.38, 0.28, 0.24, 0.10])
paperless_billing = np.random.choice(["Yes", "No"], size=n, p=[0.65, 0.35])
senior_citizen = np.random.choice([0, 1], size=n, p=[0.83, 0.17])

total_charges = tenure_months * monthly_charges + np.random.normal(0, 120, n)
total_charges = np.round(np.clip(total_charges, 0, None), 2)

# Build a churn probability using realistic business logic.
risk = (
    -2.2
    + 0.030 * monthly_charges
    - 0.035 * tenure_months
    + 0.35 * support_tickets
    + 0.45 * late_payments
    + 0.90 * (contract_type == "Month-to-month")
    - 0.60 * (contract_type == "Two year")
    + 0.50 * (internet_service == "Fiber optic")
    + 0.45 * (payment_method == "Electronic check")
    + 0.25 * (paperless_billing == "Yes")
    + 0.30 * senior_citizen
)

churn_probability = 1 / (1 + np.exp(-risk))
churn = np.random.binomial(1, churn_probability)

df = pd.DataFrame({
    "customer_id": range(10001, 10001 + n),
    "tenure_months": tenure_months,
    "monthly_charges": monthly_charges,
    "total_charges": total_charges,
    "support_tickets": support_tickets,
    "late_payments": late_payments,
    "contract_type": contract_type,
    "internet_service": internet_service,
    "payment_method": payment_method,
    "paperless_billing": paperless_billing,
    "senior_citizen": senior_citizen,
    "churn": churn
})

df.head()

## 3. Initial Data Inspection

This step checks the shape of the dataset, data types, missing values, and the churn distribution.

In [ ]:
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nChurn distribution:")
print(df["churn"].value_counts(normalize=True).round(3))

## 4. Exploratory Data Analysis

The purpose of EDA is to understand which customer characteristics appear related to churn before building the model.

In [ ]:
churn_counts = df["churn"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(["Stayed", "Churned"], churn_counts.values)
plt.title("Customer Churn Distribution")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
contract_churn = df.groupby("contract_type")["churn"].mean().sort_values(ascending=False)

plt.figure(figsize=(7, 4))
plt.bar(contract_churn.index, contract_churn.values)
plt.title("Churn Rate by Contract Type")
plt.ylabel("Churn Rate")
plt.xticks(rotation=20)
plt.show()

contract_churn

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df.loc[df["churn"] == 0, "tenure_months"], bins=25, alpha=0.7, label="Stayed")
plt.hist(df.loc[df["churn"] == 1, "tenure_months"], bins=25, alpha=0.7, label="Churned")
plt.title("Tenure Distribution by Churn Status")
plt.xlabel("Tenure in Months")
plt.ylabel("Number of Customers")
plt.legend()
plt.show()

In [ ]:
numeric_cols = ["tenure_months", "monthly_charges", "total_charges", "support_tickets", "late_payments", "senior_citizen", "churn"]
corr = df[numeric_cols].corr()["churn"].sort_values(ascending=False)
corr

## 5. Feature Engineering

Feature engineering creates new variables that may help the model find stronger patterns. Here we create:

- `average_charge_per_month`
- `high_support_flag`
- `late_payment_flag`
- `new_customer_flag`

These features are easy to explain in a business setting.

In [ ]:
model_df = df.copy()

model_df["average_charge_per_month"] = model_df["total_charges"] / model_df["tenure_months"]
model_df["high_support_flag"] = (model_df["support_tickets"] >= 3).astype(int)
model_df["late_payment_flag"] = (model_df["late_payments"] >= 1).astype(int)
model_df["new_customer_flag"] = (model_df["tenure_months"] <= 6).astype(int)

model_df.head()

## 6. Prepare Features and Target

We remove `customer_id` because it is only an identifier. The model should learn from customer behavior and account characteristics, not from ID numbers.

In [ ]:
target = "churn"

X = model_df.drop(columns=["customer_id", target])
y = model_df[target]

categorical_features = X.select_dtypes(include="object").columns.tolist()
numerical_features = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)

## 7. Train/Test Split

The model is trained on one part of the dataset and tested on a separate part. Stratification keeps the churn proportion similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

## 8. Build Preprocessing Pipeline

Numerical variables are scaled for logistic regression. Categorical variables are converted into dummy variables using one-hot encoding.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

## 9. Train Logistic Regression Model

Logistic regression is a strong baseline for churn prediction because it estimates the probability that a customer will churn.

In [ ]:
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]

print("Logistic Regression Classification Report:")
print(classification_report(y_test, logistic_pred))

## 10. Train Random Forest Model

Random forest can capture nonlinear patterns and interactions. For example, high monthly charges may be more risky for newer customers than for long-term customers.

In [ ]:
forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=8,
        random_state=42,
        class_weight="balanced"
    ))
])

forest_model.fit(X_train, y_train)

forest_pred = forest_model.predict(X_test)
forest_prob = forest_model.predict_proba(X_test)[:, 1]

print("Random Forest Classification Report:")
print(classification_report(y_test, forest_pred))

## 11. Compare Model Performance

For churn prediction, accuracy alone is not enough. Recall is especially important because the business usually wants to identify as many at-risk customers as possible before they leave.

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1 Score": f1_score(y_true, y_pred),
        "ROC AUC": roc_auc_score(y_true, y_prob)
    }

results = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, logistic_pred, logistic_prob),
    evaluate_model("Random Forest", y_test, forest_pred, forest_prob)
])

results.round(4)

In [ ]:
plt.figure(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, logistic_prob, name="Logistic Regression")
RocCurveDisplay.from_predictions(y_test, forest_prob, name="Random Forest")
plt.title("ROC Curve Comparison")
plt.show()

## 12. Confusion Matrix

The confusion matrix shows correct and incorrect predictions. In churn analysis:

- False negatives are customers who were predicted to stay but actually churned
- False positives are customers predicted to churn but actually stayed

False negatives can be costly because the company misses a chance to intervene.

In [ ]:
best_model_name = results.sort_values("ROC AUC", ascending=False).iloc[0]["Model"]
best_pred = forest_pred if best_model_name == "Random Forest" else logistic_pred

cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title(f"Confusion Matrix: {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.xticks([0, 1], ["Stayed", "Churned"])
plt.yticks([0, 1], ["Stayed", "Churned"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

cm

## 13. Feature Importance

Feature importance helps explain which variables are most useful for predicting churn. This is especially useful for business interpretation.

In [ ]:
trained_preprocessor = forest_model.named_steps["preprocessor"]
trained_forest = forest_model.named_steps["classifier"]

encoded_cat_features = trained_preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features)
all_feature_names = numerical_features + encoded_cat_features.tolist()

importance = pd.DataFrame({
    "feature": all_feature_names,
    "importance": trained_forest.feature_importances_
}).sort_values("importance", ascending=False)

importance.head(15)

In [ ]:
top_importance = importance.head(12).sort_values("importance")

plt.figure(figsize=(8, 6))
plt.barh(top_importance["feature"], top_importance["importance"])
plt.title("Top Churn Prediction Features")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 14. Customer Churn Risk Scoring

A churn model becomes more useful when it produces customer-level risk scores. The company can use these scores to prioritize retention efforts.

In [ ]:
best_model = forest_model if best_model_name == "Random Forest" else logistic_model

score_table = model_df.loc[X_test.index, [
    "customer_id",
    "tenure_months",
    "monthly_charges",
    "contract_type",
    "support_tickets",
    "late_payments",
    "churn"
]].copy()

score_table["predicted_churn_probability"] = best_model.predict_proba(X_test)[:, 1]
score_table["risk_level"] = pd.cut(
    score_table["predicted_churn_probability"],
    bins=[0, 0.35, 0.65, 1],
    labels=["Low", "Medium", "High"]
)

score_table.sort_values("predicted_churn_probability", ascending=False).head(15)

## 15. Business Interpretation

The churn model can help a company identify customers who are likely to leave. The most useful business action is not simply predicting churn, but acting on the prediction.

Possible actions include:

- Offering discounts to high-risk customers
- Contacting customers with many support tickets
- Improving onboarding for new customers
- Creating loyalty offers for month-to-month customers
- Monitoring customers with late payments or high monthly charges

The model should be evaluated not only by statistical performance, but also by whether it helps reduce actual churn and improve customer retention.

## 16. Conclusion

This notebook demonstrated a complete customer churn prediction analysis. The project started with customer-level data, explored churn patterns, engineered useful features, trained classification models, compared performance, interpreted feature importance, and created a customer-level risk scoring table.

The main lesson is that churn prediction is both a technical and business problem. A good model should identify at-risk customers accurately, but the company must also design meaningful retention strategies based on the model's output.

For a stronger real-world version, the analysis could be expanded with real customer interaction logs, product usage data, customer satisfaction scores, marketing history, and time-based churn tracking.